# PilotGuard — Full Training Pipeline (Colab Pro + H100)

This notebook runs the entire ML training pipeline:
1. Upload project from Google Drive
2. Install dependencies
3. Pre-compute DINOv2 features (GPU)
4. Train XGBoost geometric classifier (CPU)
5. Train DINOv2 classification heads (GPU)
6. Download trained models

**Runtime**: GPU → H100 | High RAM enabled

---

## 0. Verify GPU & Mount Drive

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {total_mem / 1024**3:.1f} GB")
    print(f"bf16 support: {torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → H100")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Upload Project to Google Drive

**Before running this notebook**, upload your project to Google Drive:

### Code (zip — small):
On your PC in PowerShell:
```powershell
Compress-Archive -Path D:\PilotGuard\backend, D:\PilotGuard\configs, D:\PilotGuard\scripts -DestinationPath D:\PilotGuard\pilotguard_code.zip
```
Upload `pilotguard_code.zip` to **Google Drive** → `My Drive/PilotGuard/`

### Data (direct upload — too large to zip):
1. Install **Google Drive for Desktop** (if not already): [download](https://www.google.com/intl/en/drive/download/)
2. It mounts as a drive letter (e.g., `G:\`)
3. Run in PowerShell:
```powershell
mkdir "G:\My Drive\PilotGuard\data\processed" -Force
robocopy "D:\PilotGuard\data\processed" "G:\My Drive\PilotGuard\data\processed" /E /MT:8 /R:3
```
This copies ~35 GB of processed data directly — no zipping needed.

Then run the cells below.

In [ ]:
import os
import shutil

DRIVE_DIR = "/content/drive/MyDrive/PilotGuard"
WORK_DIR = "/content/pilotguard"

# Clean previous run if exists
if os.path.exists(WORK_DIR):
    print(f"Removing existing {WORK_DIR}...")
    shutil.rmtree(WORK_DIR)

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Work directory: {WORK_DIR}")
print(f"Drive directory: {DRIVE_DIR}")
print(f"Drive contents: {os.listdir(DRIVE_DIR)}")

In [ ]:
%%time
# Extract code from zip
!cd /content && unzip -qo "{DRIVE_DIR}/pilotguard_code.zip" -d {WORK_DIR}
print("Code extracted!")
print(os.listdir(WORK_DIR))

In [ ]:
%%time
import glob

# Copy processed data from Drive (uploaded directly, not zipped)
data_src = f"{DRIVE_DIR}/data/processed"
data_dst = f"{WORK_DIR}/data/processed"

if os.path.exists(data_src):
    print(f"Copying data from Drive: {data_src}")
    print("This may take a while for ~35 GB...")
    !cp -r "{data_src}" "{WORK_DIR}/data/"
    print("Data copied!")
else:
    # Fallback: check if data zip exists
    zip_path = f"{DRIVE_DIR}/pilotguard_data.zip"
    if os.path.exists(zip_path):
        print(f"Extracting data zip: {zip_path}")
        !cd /content && unzip -qo "{zip_path}" -d {WORK_DIR}
        print("Data extracted from zip!")
    else:
        raise FileNotFoundError(
            f"No data found! Upload either:\n"
            f"  1. data/processed/ folder to {DRIVE_DIR}/data/processed/\n"
            f"  2. pilotguard_data.zip to {DRIVE_DIR}/"
        )

# Verify manifests
manifests = glob.glob(f"{WORK_DIR}/data/processed/*_manifest.csv")
print(f"\nFound {len(manifests)} manifests:")
for m in manifests:
    lines = sum(1 for _ in open(m)) - 1
    print(f"  {os.path.basename(m)}: {lines:,} samples")

## 2. Install Dependencies

In [ ]:
%%time
os.chdir(f"{WORK_DIR}/backend")
!pip install -q -e ".[dev]" 2>&1 | tail -5
print("\nDependencies installed!")

In [ ]:
# Verify imports
import torch
import xgboost
import sklearn
import yaml
import cv2
import numpy as np

print(f"torch:    {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"xgboost:  {xgboost.__version__}")
print(f"sklearn:  {sklearn.__version__}")
print(f"opencv:   {cv2.__version__}")
print(f"numpy:    {np.__version__}")
print("\nAll imports OK!")

## 3. Load Config & Set Seeds

In [ ]:
import sys
import random
import logging

os.chdir(f"{WORK_DIR}/backend")
sys.path.insert(0, f"{WORK_DIR}/backend")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pilotguard")

# Load config
with open(f"{WORK_DIR}/configs/training_config.yaml") as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Seed: {SEED}")
print(f"Device: {DEVICE}")
print(f"Profiles: {list(config['training']['profiles'].keys())}")

## 4. Stage 1 — Pre-compute DINOv2 Features (GPU)

This extracts 384-dim features from DINOv2 ViT-S/14 for every image.
Features are cached to disk so training heads is fast.

**~2-3 hours for all datasets on H100.**

In [ ]:
from scripts.train_hpc import precompute_dinov2_features, load_manifest

profiles = config["training"]["profiles"]

for name, profile in profiles.items():
    print(f"\n{'='*60}")
    print(f"Pre-computing features: {name}")
    print(f"{'='*60}")
    precompute_dinov2_features(name, profile, DEVICE)

In [ ]:
# Verify cached features
feature_files = glob.glob(f"{WORK_DIR}/data/features/*/dinov2_features.npz")
for f in feature_files:
    data = np.load(f)
    print(f"{os.path.basename(os.path.dirname(f))}: {len(data['features']):,} features, shape={data['features'].shape}")

## 5. Stage 2 — Train XGBoost Geometric Classifier (CPU)

Lightweight baseline: 7 geometric features → XGBoost → drowsy/alert.

**~30-60 min (feature extraction is the bottleneck, CPU-bound).**

In [ ]:
from scripts.train_hpc import train_xgboost_model

train_xgboost_model(config)

## 6. Stage 3 — Train DINOv2 Classification Heads (GPU)

Trains classification heads on cached DINOv2 features with:
- **Linear warmup** → **cosine annealing** LR schedule
- Mixed precision (bf16 on H100)
- Early stopping on validation F1

Run each profile one at a time so you can monitor progress.

In [ ]:
from scripts.train_hpc import train_dinov2_profile

# --- NTHU-DDD: Binary drowsiness (50 epochs, 5 warmup) ---
train_dinov2_profile("nthu_drowsiness", profiles["nthu_drowsiness"], config, DEVICE)

In [ ]:
# --- UTA-RLDD: Binary drowsiness (40 epochs, 3 warmup) ---
train_dinov2_profile("uta_drowsiness", profiles["uta_drowsiness"], config, DEVICE)

In [ ]:
# --- AffectNet: Emotion classification (60 epochs, 5 warmup) ---
train_dinov2_profile("affectnet_emotion", profiles["affectnet_emotion"], config, DEVICE)

In [ ]:
# --- DISFA: Action unit classification (50 epochs, 5 warmup) ---
train_dinov2_profile("disfa_au", profiles["disfa_au"], config, DEVICE)

## 7. Review Results

In [ ]:
import json

print("=" * 70)
print("TRAINING RESULTS SUMMARY")
print("=" * 70)

# Check all saved metadata
meta_files = glob.glob(f"{WORK_DIR}/backend/models/**/*_metadata.json", recursive=True)
for mf in sorted(meta_files):
    with open(mf) as f:
        meta = json.load(f)
    name = os.path.basename(mf).replace("_metadata.json", "")
    print(f"\n--- {name} ---")
    if "metrics" in meta:
        m = meta["metrics"]
    elif "test_metrics" in meta:
        m = meta["test_metrics"]
    else:
        m = {}
    for k, v in m.items():
        print(f"  {k}: {v:.4f}")
    if "best_val_f1" in meta:
        print(f"  best_val_f1: {meta['best_val_f1']:.4f} (epoch {meta.get('epochs_trained', '?')})")
    print(f"  train_samples: {meta.get('train_samples', '?')}")

## 8. TensorBoard (Optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {WORK_DIR}/backend/runs

## 9. Save Models to Google Drive

Copy trained models back to Drive so they persist after the Colab session ends.

In [ ]:
import shutil

# Save models
drive_models = f"{DRIVE_DIR}/trained_models"
if os.path.exists(drive_models):
    shutil.rmtree(drive_models)
shutil.copytree(f"{WORK_DIR}/backend/models", drive_models)
print(f"Models saved to Drive: {drive_models}")

# Save TensorBoard runs
drive_runs = f"{DRIVE_DIR}/runs"
if os.path.exists(drive_runs):
    shutil.rmtree(drive_runs)
shutil.copytree(f"{WORK_DIR}/backend/runs", drive_runs)
print(f"TensorBoard logs saved to Drive: {drive_runs}")

# Save feature cache (so you don't need to re-extract next time)
drive_features = f"{DRIVE_DIR}/features"
if os.path.exists(drive_features):
    shutil.rmtree(drive_features)
shutil.copytree(f"{WORK_DIR}/data/features", drive_features)
print(f"Feature cache saved to Drive: {drive_features}")

print("\nAll results saved to Google Drive!")

## 10. Download Models to Local PC

Run this cell to download a zip of all trained models directly to your browser.

In [ ]:
# Zip models for download
shutil.make_archive("/content/pilotguard_models", "zip", f"{WORK_DIR}/backend/models")

from google.colab import files
files.download("/content/pilotguard_models.zip")
print("Download started! Save to D:\\PilotGuard\\backend\\models\\ and unzip.")